# Strawberry shelf life vs storage temperature

**Source:** Yan et al., "Effect of Different Storage Temperature on Respiration Rate and Quality of Strawberry", *Journal of Refrigeration* (制冷学报), Issue 5, pp. 37-42.

Measured shelf life: 9 days at 0°C, 7 days at 5°C, 5 days at 10°C, 4 days at 15°C, 3 days at 20°C, 2 days at 25°C, 8 hours at 30°C.

**Cross-check:** the University of South Florida Food Quality Lab reports strawberry postharvest life of more than 9 days at a constant 1.5°C.

In [ ]:
# Cell 1: the published measurements
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

shelf_life = pd.DataFrame({
    "temp_c": [0, 5, 10, 15, 20, 25, 30],
    "shelf_life_days": [9, 7, 5, 4, 3, 2, 8 / 24],  # 30°C: 8 hours
})
print(shelf_life.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(shelf_life["temp_c"], shelf_life["shelf_life_days"], marker="o")
ax.set_title("Strawberry shelf life vs storage temperature (Yan et al.)")
ax.set_xlabel("Storage temperature (°C)")
ax.set_ylabel("Shelf life (days)")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 2: shelf life at any temperature, by connecting the measured dots with straight lines
def shelf_life_days_at(temp_c):
    """Shelf life in days at a constant temperature (works on one number or an array)."""
    # np.interp holds the end values outside the data: below 0°C -> the 0°C value, above 30°C -> the
    # 30°C value. These are limits, not measurements: the study didn't test outside 0-30°C.
    return np.interp(temp_c, shelf_life["temp_c"], shelf_life["shelf_life_days"])


for t in [0, 1, 2, 3, 4, 8, 12, 30]:
    print(f"{t:>3} °C: {shelf_life_days_at(t):.2f} days")

In [ ]:
# Cell 3: shelf life used up during a trip, treating shelf life as a budget
def shelf_life_remaining_pct(temps_c, hours_per_step):
    """% of shelf life left after a series of equal time steps at the given temperatures."""
    temps_c = np.asarray(temps_c, dtype=float)
    # Each step uses (its length) / (total life at that temperature) of the fruit's life
    used = (hours_per_step / (shelf_life_days_at(temps_c) * 24)).sum()
    return max(0.0, 100 * (1 - used))


steps_per_day = 24  # 1-hour steps for the test cases
tests = {
    "5 days at 0°C (expect ~44%)": [0] * 5 * steps_per_day,
    "5 days at 2°C": [2] * 5 * steps_per_day,
    "5 days at 10°C (expect 0%)": [10] * 5 * steps_per_day,
    "4 days at 2°C + 6 hours at 15°C": [2] * 4 * steps_per_day + [15] * 6,
}
for name, temps in tests.items():
    print(f"{name:<34} {shelf_life_remaining_pct(temps, hours_per_step=1):5.1f}% left")

assert abs(shelf_life_remaining_pct([0] * 120, 1) - 100 * (1 - 5 / 9)) < 1e-9
assert shelf_life_remaining_pct([10] * 120, 1) < 1e-9  # not exactly 0: tiny rounding in the sum

In [ ]:
# Cell 4: shelf life left at the end of each real sensor's data
from pathlib import Path

clean = pd.read_csv(Path("../data/processed/strawberry_clean.csv"), parse_dates=["time"])
full_sensors = clean[~clean["partial_sensor"]]  # handling_event rows kept: the fruit lived through it


def remaining_from_readings(sensor_rows):
    """Average one sensor's readings into 15-minute blocks, then apply the shelf-life budget."""
    blocks = sensor_rows.set_index("time")["temp_c"].resample("15min").mean().dropna()
    return shelf_life_remaining_pct(blocks.values, hours_per_step=0.25)


rows = []
for (shipment, pallet, layer), sensor_df in full_sensors.groupby(["shipment", "pallet", "layer"]):
    rows.append({
        "shipment": shipment,
        "pallet_layer": f"{pallet}_{layer}",
        "transit_only_pct": remaining_from_readings(sensor_df[sensor_df["phase"] == "transit"]),
        "full_window_pct": remaining_from_readings(sensor_df),  # precool + transit
    })
per_sensor = pd.DataFrame(rows)

summary = per_sensor.groupby("shipment").agg(
    transit_mean=("transit_only_pct", "mean"),
    transit_worst=("transit_only_pct", "min"),
    full_mean=("full_window_pct", "mean"),
    full_worst=("full_window_pct", "min"),
)
print("Shelf life remaining (%) at the end of each sensor's data:")
print(summary.round(1).to_string())